<a href="https://colab.research.google.com/github/deepan98raj-dotcom/My_Project/blob/main/Deploy_ML_Models_as_Service_with_FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install colabcode
!pip install fastapi

Requested uvicorn==0.13.1 from https://files.pythonhosted.org/packages/ef/67/546c35e9fffb585ea0608ba3bdcafe17ae402e304367203d0b08d6c23051/uvicorn-0.13.1-py3-none-any.whl (from colabcode) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    python-dotenv (>=0.13.*) ; extra == 'standard'
                   ~~~~~~~^
Please use pip<24.1 if you need to use this version.
INFO: pip is looking at multiple versions of colabcode to determine which version is compatible with other requirements. This could take a while.
  Using cached uvicorn-0.13.1-py3-none-any.whl.metadata (4.6 kB)
Requested uvicorn==0.13.1 from https://files.pythonhosted.org/packages/ef/67/546c35e9fffb585ea0608ba3bdcafe17ae402e304367203d0b08d6c23051/uvicorn-0.13.1-py3-none-any.whl (from colabcode) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    python-dotenv (>=0.13.*) ; extra == 'standard'
                   ~~~~~~~^
Please use pip<24.1 if you need to use this versio

In [ ]:
from colabcode import ColabCode
from fastapi import FastAPI

In [ ]:
cc = ColabCode(port=12000, code=False)

In [ ]:
app = FastAPI()

@app.get("/")
async def read_root():
  return {"message": "Subscribe to @1littlecoder"}

In [ ]:
cc.run_app(app=app)

Public URL: NgrokTunnel: "http://ebd7bd6cabe1.ngrok.io" -> "http://localhost:12000"


INFO:     Started server process [63]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:12000 (Press CTRL+C to quit)


INFO:     103.25.46.30:0 - "GET / HTTP/1.1" 200 OK
INFO:     103.25.46.30:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [63]


In [ ]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pickle

iris = load_iris()
model = GaussianNB()

X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.1)
model_f = model.fit(X_train, y_train)

print("Model score: ", model.score(X_train, y_train))
print("Test Accuracy: ", model.score(X_test, y_test))

pickle.dump(model_f, open("model_gb.pkl", "wb"))

Model score:  0.9629629629629629
Test Accuracy:  0.9333333333333333


In [ ]:
%%writefile models.py
from pydantic import BaseModel, conlist
from typing import List


class Iris(BaseModel):
    data: List[conlist(float, min_items=4, max_items=4)]

Writing models.py


In [ ]:
import pickle
import logging
from fastapi import FastAPI
from models import Iris

app = FastAPI(title="ML Models as API on Google Colab", description="with FastAPI and ColabCode", version="1.0")

# # Initialize logging
# my_logger = logging.getLogger()
# my_logger.setLevel(logging.DEBUG)
# logging.basicConfig(level=logging.DEBUG, filename='logs.log')

model = None

@app.on_event("startup")
def load_model():
    global model
    model = pickle.load(open("model_gb.pkl", "rb"))

@app.post("/api", tags=["prediction"])
async def get_predictions(iris: Iris):
    try:
        data = dict(iris)['data']
        print(data)
        iris_types = {
            0: 'setosa',
            1: 'versicolor',
            2: 'virginica'
        }
        prediction = list(map(lambda x: iris_types[x], model.predict(data).tolist()))
        log_proba = model.predict_log_proba(data).tolist()
        return {"prediction": prediction, "log_proba": log_proba}
    except:
        my_logger.error("Something went wrong!")
        return {"prediction": "error"}

In [ ]:
cc.run_app(app=app)

Public URL: NgrokTunnel: "http://a7ade6155418.ngrok.io" -> "http://localhost:12000"


INFO:     Started server process [63]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:12000 (Press CTRL+C to quit)


INFO:     103.25.46.30:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     103.25.46.30:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     103.25.46.30:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     103.25.46.30:0 - "GET /openapi.json HTTP/1.1" 200 OK
[[4.5, 1.5, 3.5, 2.5]]
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 200 OK
[[1.5, 3.5, 3.5, 2.5]]
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 200 OK
[[1.5, 5.5, 9.5, 2.5]]
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 200 OK
[[1.5, 5.5, 9.5, 2.5]]
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 200 OK
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 422 Unprocessable Entity
[[1.5, 5.5, 9.5, 2.5]]
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 200 OK
[[1.5, 5.5, 9.5, 2.5]]
INFO:     103.25.46.30:0 - "POST /api HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [63]
